# L22 · Policy learning con presupuesto

**S22 · Lectura: Cap. 21 · 40–55 min**

## La pregunta

¿A quién tratar si solo podemos atender al 20 %?

## Qué harás

Construir y comparar una política causal bajo presupuesto.

- Lee la idea breve.
- Ejecuta la demostración y mira el gráfico.
- Completa las celdas marcadas.
- Escribe qué significa el resultado y qué no permite afirmar.

**Producto:** Policy top-k, valor y comparación con baselines.

> **Idea clave:** El mejor ranking de riesgo no siempre es la mejor política de intervención.

La solución está en `la ruta de rescate docente (no incluida)`. Intenta primero el notebook del estudiante.

In [ ]:
# Localiza lab_common.py (carpeta actual o una superior)
import sys, pathlib
for _p in [pathlib.Path.cwd(), *list(pathlib.Path.cwd().parents)[:3]]:
    if (_p / "lab_common.py").exists():
        sys.path.insert(0, str(_p)); break

## 0. Preparación

In [ ]:
from lab_common import *
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor, export_text
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.linear_model import LogisticRegression
rng = setup()
sc = load_scenario("real/ihdp"); sc.card()
cols = list(sc.covariates)
Xtr, Ytr, Ttr = sc.xyt("train"); Ttr = Ttr.astype(int)
Xv, Yv, Tv = sc.xyt("val"); Xt, Yt, Tt = sc.xyt("test")
XH = np.vstack([Xv, Xt]); YH = np.r_[Yv, Yt]; TH = np.r_[Tv, Tt].astype(int)          # HOLD-OUT = validación + prueba
print("entrenamiento:", len(Ytr), "· hold-out (validación + prueba):", len(YH))

In [ ]:
# Vista rápida de los datos del escenario
import matplotlib.pyplot as plt
import numpy as np
_fig, _ax = plt.subplots(1, 2, figsize=(9, 3.2))
try:
    _scenario = globals().get("sc") or globals().get("lc")
    if _scenario is None:
        raise NameError("No hay un escenario cargado para el gráfico editorial")
    _df = _scenario.train
    if _scenario.treatment is not None and _scenario.outcome is not None:
        _t = _df[_scenario.treatment].to_numpy(float)
        _y = _df[_scenario.outcome].to_numpy(float)
        if np.unique(_t).size == 2:
            _counts = [int((_t == value).sum()) for value in (0, 1)]
            _ax[0].bar(["control", "tratado"], _counts)
            _ax[0].set_title("Asignación")
            _ax[1].boxplot([_y[_t == 0], _y[_t == 1]], labels=["control", "tratado"])
            _ax[1].set_title("Resultado observado")
        else:
            _ax[0].hist(_t, bins=20)
            _ax[0].set_title(_scenario.treatment)
            _ax[1].scatter(_t, _y, s=8, alpha=.35)
            _ax[1].set_title(f"{_scenario.treatment} frente a {_scenario.outcome}")
    elif _scenario.adj is not None:
        _ax[0].imshow(_scenario.adj, cmap="Greys", interpolation="nearest")
        _ax[0].set_title("Grafo del escenario")
        _ax[1].axis("off")
    else:
        _ax[0].axis("off"); _ax[1].axis("off")
    _fig.suptitle(f"Escenario: {_scenario.name}")
    _fig.tight_layout(); plt.show()
finally:
    plt.close(_fig)

## 1. Del CATE a una política con presupuesto (§21.2, §21.3)
Valor de una política: $V(\pi)=E[Y(\pi(X))]=E[\mu_0(X)+\pi(X)\tau(X)]$. La política óptima sin restricciones es $\mathbf 1\{\tau(x)>0\}$. Con un presupuesto $\beta$ (aquí tratar a lo sumo el **30 %**), se trata a quienes tienen el $\hat\tau$ más alto.

Primero estimamos nuisance y CATE **solo con el conjunto de entrenamiento**.

In [ ]:
ETR = lambda: ExtraTreesRegressor(n_estimators=40, min_samples_leaf=8, max_features=0.8, random_state=0)
def psi_cf(X, y, d, K=2, seed=SEED):
    n = len(y); psi = np.zeros(n)
    for te in kfold(n, K, seed):
        tr = np.setdiff1d(np.arange(n), te); t1, t0 = tr[d[tr] == 1], tr[d[tr] == 0]
        m1 = ETR().fit(X[t1], y[t1]).predict(X[te]); m0 = ETR().fit(X[t0], y[t0]).predict(X[te])
        e = np.clip(LogisticRegression(max_iter=500).fit(X[tr], d[tr]).predict_proba(X[te])[:, 1], .05, .95)
        psi[te] = m1 - m0 + d[te] * (y[te] - m1) / e - (1 - d[te]) * (y[te] - m0) / (1 - e)
    return psi
psi_tr = psi_cf(Xtr, Ytr, Ttr)
cate_model = ETR().fit(Xtr, psi_tr)                                    # DR-learner: regresión de las puntuaciones AIPW
tau_tr, tau_H = cate_model.predict(Xtr), cate_model.predict(XH)
# nuisance para evaluar en hold-out (entrenados SOLO con train)
mu1_m = ETR().fit(Xtr[Ttr == 1], Ytr[Ttr == 1]); mu0_m = ETR().fit(Xtr[Ttr == 0], Ytr[Ttr == 0])
e_m = LogisticRegression(max_iter=500).fit(Xtr, Ttr)
mu1_H, mu0_H = mu1_m.predict(XH), mu0_m.predict(XH); e_H = np.clip(e_m.predict_proba(XH)[:, 1], .05, .95)
BETA = 0.30
def top_frac(score, beta=BETA):
    k = int(np.floor(beta * len(score))); pi = np.zeros(len(score), int); pi[np.argsort(-score)[:k]] = 1; return pi
pi_top = top_frac(tau_H)
print(f"política top-{int(BETA*100)} % por τ̂: trata al {100 * pi_top.mean():.1f} % del hold-out")

## 2. Árbol de política (profundidad 3) y presupuesto (Alg. 9)
Se construye un árbol de profundidad 3 con **objetivo $\hat\tau$** y se trata a las hojas cuyo $\hat\tau$ medio supera el umbral $c$ = cuantil 70 % de los $\hat\tau$ de entrenamiento. Esto produce **reglas legibles**, a costa de aproximar la regla óptima.

In [ ]:
arbol = DecisionTreeRegressor(max_depth=3, min_samples_leaf=25, random_state=0).fit(Xtr, tau_tr)
c = np.quantile(tau_tr, 1 - BETA)
trata_hoja = {h: arbol.tree_.value[h][0][0] > c for h in range(arbol.tree_.node_count) if arbol.tree_.children_left[h] == -1}
pi_tree = np.array([trata_hoja[h] for h in arbol.apply(XH)]).astype(int)
print(export_text(arbol, feature_names=cols, max_depth=3))
print(f"fracción tratada por el árbol en hold-out: {100 * pi_tree.mean():.1f} %  (presupuesto: {int(BETA * 100)} %)")

## 3. Evaluar **con** y **sin** la verdad
En simulación (IHDP es semisintético) conocemos $\tau_i$ y podemos medir el *welfare* oráculo $E[\pi\tau]$. Con datos reales **no existe**: solo se puede **estimar** el valor de la política con los datos observados.

Tres estimadores de $V(\pi)$ sobre el hold-out ($e_\pi=e$ si $\pi=1$, $1-e$ si $\pi=0$):
- **IPS:** $\frac1n\sum \mathbf 1\{T_i=\pi_i\}Y_i/e_\pi$;
- **SNIPS:** IPS normalizado por la suma de pesos;
- **Doblemente robusto (DR):** $\frac1n\sum[\hat\mu_\pi+\mathbf 1\{T=\pi\}(Y-\hat\mu_\pi)/e_\pi]$.

### Completa tú
`valor_politica_dr(pi, y, t, mu0, mu1, e)` → `(V, se, psi)` con `psi` el vector de puntuaciones (su desviación da el error estándar por función de influencia).

In [ ]:
def valor_ips(pi, y, t, e):
    e_pi = np.where(pi == 1, e, 1 - e); return float(np.mean((t == pi) * y / e_pi))
def valor_snips(pi, y, t, e):
    e_pi = np.where(pi == 1, e, 1 - e); w = (t == pi) / e_pi; return float((w * y).sum() / w.sum())

In [ ]:
# ✏️ Completa tú — valor_politica_dr
# Pista: mu_pi = mu1 donde pi==1 y mu0 donde pi==0; e_pi = e donde pi==1 y 1-e donde pi==0; psi = mu_pi + 1{t==pi}/e_pi * (y - mu_pi).
def valor_politica_dr(pi, y, t, mu0, mu1, e):
    """Valor DR de la política pi. Devuelve (V, se, psi)."""
    raise NotImplementedError("completa valor_politica_dr")

In [ ]:
# caso de verificación: mu = 0, e = 0,5, pi = t  →  psi_i = y_i / 0,5
y_, t_ = np.array([1., 2.]), np.array([1, 0])
lab_check("valor_politica_dr", lambda: (valor_politica_dr(np.array([1, 0]), y_, t_, np.zeros(2), np.zeros(2), np.full(2, .5))[0], valor_politica_dr(np.array([1, 1]), y_, t_, np.zeros(2), np.zeros(2), np.full(2, .5))[0]),
          [((), (3.0, 1.0), 1e-9)])

## Sección de evaluación reservada

La edición pública termina antes de las celdas docentes que comparan con verdad simulada, efectos potenciales o resultados oráculo. Esa revisión se ejecuta aparte; aquí debes sostener la decisión con observables, incertidumbre y supuestos declarados.
